- in this notebook we will learn the how to build the persistance term how the persistance work and the build the worklflow 
- in persistance work implemented the joke creation chatbot on different topics and how to build the end to end workflow

- in this workflow persistance main points arethe memory ,checkpoints ,superstep ,node and threads

- benfits of persisitance is 1.short term memory  2. fault tolerance 3.human in the loop 4 .time travel 

In [37]:
import os 
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel,Field
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage,BaseMessage
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver,MemorySaver





In [38]:
load_dotenv()
llm=ChatGoogleGenerativeAI( model="gemini-2.5-flash",api_key=os.getenv("GOOGLE_API_KEY"))


In [39]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [40]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [41]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [42]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [43]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza tell so many bad jokes?\n\nBecause it was so **cheesy**!',
 'explanation': 'This joke is a classic example of a **pun**, which relies on the two different meanings of the word "cheesy."\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (for pizza):** When we talk about pizza, "cheesy" literally means it contains a lot of cheese, or is covered generously in cheese. This is generally a good thing for pizza!\n\n2.  **Figurative Meaning (for jokes/humor):** When we describe a joke, comment, or even a song or movie as "cheesy," it means it\'s:\n    *   **Unoriginal or predictable.**\n    *   **Overly sentimental or mawkish.**\n    *   **Trying too hard to be funny or clever, but failing.**\n    *   **Corny or cringeworthy.**\n    *   Basically, a "cheesy joke" is a bad, groan-inducing joke.\n\nThe humor comes from the **double meaning**. The pizza *literally* has a lot of cheese, making it "cheesy" in the food sense. The joke then cleverly app

In [44]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza tell so many bad jokes?\n\nBecause it was so **cheesy**!', 'explanation': 'This joke is a classic example of a **pun**, which relies on the two different meanings of the word "cheesy."\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (for pizza):** When we talk about pizza, "cheesy" literally means it contains a lot of cheese, or is covered generously in cheese. This is generally a good thing for pizza!\n\n2.  **Figurative Meaning (for jokes/humor):** When we describe a joke, comment, or even a song or movie as "cheesy," it means it\'s:\n    *   **Unoriginal or predictable.**\n    *   **Overly sentimental or mawkish.**\n    *   **Trying too hard to be funny or clever, but failing.**\n    *   **Corny or cringeworthy.**\n    *   Basically, a "cheesy joke" is a bad, groan-inducing joke.\n\nThe humor comes from the **double meaning**. The pizza *literally* has a lot of cheese, making it "cheesy" in the food sense. The jok

In [45]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza tell so many bad jokes?\n\nBecause it was so **cheesy**!', 'explanation': 'This joke is a classic example of a **pun**, which relies on the two different meanings of the word "cheesy."\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (for pizza):** When we talk about pizza, "cheesy" literally means it contains a lot of cheese, or is covered generously in cheese. This is generally a good thing for pizza!\n\n2.  **Figurative Meaning (for jokes/humor):** When we describe a joke, comment, or even a song or movie as "cheesy," it means it\'s:\n    *   **Unoriginal or predictable.**\n    *   **Overly sentimental or mawkish.**\n    *   **Trying too hard to be funny or clever, but failing.**\n    *   **Corny or cringeworthy.**\n    *   Basically, a "cheesy joke" is a bad, groan-inducing joke.\n\nThe humor comes from the **double meaning**. The pizza *literally* has a lot of cheese, making it "cheesy" in the food sense. The jo

In [46]:
config2={'configurable':{'thread_id':'2'}}
workflow.invoke({'topic':'pasta'},config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti break up with the meatballs?\n\nBecause it felt too *tied down*!',
 'explanation': 'This joke is a classic pun that plays on the phrase "tied down" having two different meanings:\n\n1.  **The Literal Meaning (Spaghetti & Meatballs):**\n    Imagine a plate of spaghetti and meatballs. The long strands of spaghetti often wrap themselves *around* the meatballs. They get tangled, cling to them, and can literally appear to be "tied" to them, especially when you\'re trying to pick them up with a fork.\n\n2.  **The Idiomatic/Figurative Meaning (Relationships):**\n    In human relationships, when someone says they feel "tied down," it means they feel restricted, committed, or unable to be free and independent. It implies a sense of being burdened by the relationship\'s obligations or seriousness.\n\n**The Punchline:**\nThe humor comes from applying the human relationship problem ("feeling tied down") to the spaghetti. The spaghetti, personified

In [47]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza tell so many bad jokes?\n\nBecause it was so **cheesy**!', 'explanation': 'This joke is a classic example of a **pun**, which relies on the two different meanings of the word "cheesy."\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (for pizza):** When we talk about pizza, "cheesy" literally means it contains a lot of cheese, or is covered generously in cheese. This is generally a good thing for pizza!\n\n2.  **Figurative Meaning (for jokes/humor):** When we describe a joke, comment, or even a song or movie as "cheesy," it means it\'s:\n    *   **Unoriginal or predictable.**\n    *   **Overly sentimental or mawkish.**\n    *   **Trying too hard to be funny or clever, but failing.**\n    *   **Corny or cringeworthy.**\n    *   Basically, a "cheesy joke" is a bad, groan-inducing joke.\n\nThe humor comes from the **double meaning**. The pizza *literally* has a lot of cheese, making it "cheesy" in the food sense. The jok

In [48]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza tell so many bad jokes?\n\nBecause it was so **cheesy**!', 'explanation': 'This joke is a classic example of a **pun**, which relies on the two different meanings of the word "cheesy."\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (for pizza):** When we talk about pizza, "cheesy" literally means it contains a lot of cheese, or is covered generously in cheese. This is generally a good thing for pizza!\n\n2.  **Figurative Meaning (for jokes/humor):** When we describe a joke, comment, or even a song or movie as "cheesy," it means it\'s:\n    *   **Unoriginal or predictable.**\n    *   **Overly sentimental or mawkish.**\n    *   **Trying too hard to be funny or clever, but failing.**\n    *   **Corny or cringeworthy.**\n    *   Basically, a "cheesy joke" is a bad, groan-inducing joke.\n\nThe humor comes from the **double meaning**. The pizza *literally* has a lot of cheese, making it "cheesy" in the food sense. The jo

# update State

In [49]:
workflow.update_state({"configurable":{'thread_id':'1',"checkpoint_id":'1f19e14d-43bc-6d83-8000-243845b645f1',"checkpoint_ns":""}},{'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f19e16d-1cc8-69ff-8000-c25fa9cd1564'}}

-I CREATE NEW BRANCH CHANGE INSTEAD OF PIZZA CREATE A NEW BRANCH OF SAMOSA 

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19e16d-1cc8-69ff-8000-c25fa9cd1564'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-08-22T10:47:07.150788+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f19e14d-43bc-6d83-8000-243845b645f1'}}, tasks=(PregelTask(id='d9f476a8-0f35-740d-7353-ce8fe45daff4', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza tell so many bad jokes?\n\nBecause it was so **cheesy**!', 'explanation': 'This joke is a classic example of a **pun**, which relies on the two different meanings of the word "cheesy."\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (for pizza):** When we talk about pizza, "cheesy" literally means it contain

In [54]:
workflow.invoke(None,{"configurable":{'thread_id':'1','checkpoint_id':'1f19e16d-1cc8-69ff-8000-c25fa9cd1564'}})

{'topic': 'samosa',
 'joke': 'Why did the samosa refuse to share its feelings?\n\nBecause it preferred to keep all its best *fillings* inside!',
 'explanation': 'This is a classic **pun** joke! Here\'s the breakdown:\n\n1.  **Samosa "Fillings":** A samosa is a delicious fried or baked pastry with a savory **filling** inside (often spiced potatoes, peas, onions, etc.). This filling is arguably the *best part* of the samosa – what makes it so tasty and unique.\n\n2.  **Human "Feelings":** When we talk about people, "feelings" refer to emotions, inner thoughts, and personal experiences. To "share your feelings" means to open up emotionally to others.\n\n**The Joke\'s Humor:**\n\nThe humor comes from the wordplay between "feelings" (emotions) and "fillings" (the stuff inside the samosa). They sound identical (homophones).\n\n*   The question anthropomorphizes the samosa, giving it the human ability to have and share "feelings."\n*   The punchline then cleverly pivots, using the word "filli

In [57]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa refuse to share its feelings?\n\nBecause it preferred to keep all its best *fillings* inside!', 'explanation': 'This is a classic **pun** joke! Here\'s the breakdown:\n\n1.  **Samosa "Fillings":** A samosa is a delicious fried or baked pastry with a savory **filling** inside (often spiced potatoes, peas, onions, etc.). This filling is arguably the *best part* of the samosa – what makes it so tasty and unique.\n\n2.  **Human "Feelings":** When we talk about people, "feelings" refer to emotions, inner thoughts, and personal experiences. To "share your feelings" means to open up emotionally to others.\n\n**The Joke\'s Humor:**\n\nThe humor comes from the wordplay between "feelings" (emotions) and "fillings" (the stuff inside the samosa). They sound identical (homophones).\n\n*   The question anthropomorphizes the samosa, giving it the human ability to have and share "feelings."\n*   The punchline then cleverly pivots, u

# fault Tolerance 

In [58]:
from langgraph.graph import StateGraph,END
from langgraph.checkpoint.memory import InMemorySaver
from typing  import TypedDict
import time

In [59]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str


In [60]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [61]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [62]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
❌ Kernel manually interrupted (crash simulated).


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))